In [1]:
"""
Spark-PostgreSQL Optimization: Partitioning & Batching
Learn how to optimize data transfer between Spark and PostgreSQL
"""

import os
import time
import logging
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ============================================================================
# CONFIGURATION
# ============================================================================

DB_CONFIG = {
    "url": "jdbc:postgresql://localhost:5432/Data_engineering_practice",
    "user": "postgres",
    "password": "xxxxxxxx",  # 🔴 CHANGE THIS
    "driver": "org.postgresql.Driver"
}

BASE_DIR = "/home/odinsbeard/Data_engineering_Journey/week6_spark"
JAR_PATH = f"{BASE_DIR}/jars/postgresql-42.7.1.jar"
OUTPUT_DIR = f"{BASE_DIR}/data/optimization_tests"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================================
# SETUP LOGGING
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
logger


<Logger __main__ (INFO)>

In [2]:
# ============================================================================
# INITIALIZE SPARK
# ============================================================================
def init_spark():
    """Initialize Spark session"""
    spark = SparkSession.builder \
        .appName("Spark-PostgreSQL Optimization") \
        .config("spark.jars", JAR_PATH) \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.sql.adaptive.skewJoin.enabled", "true") \
        .config("spark.sql.files.maxPartitionBytes", "128MB") \
        .getOrCreate()
    
    spark.sparkContext.setLogLevel("WARN")
    logger.info(f"✅ Spark session created (version: {spark.version})")
    return spark

# ============================================================================
# PART 1: OPTIMIZED READING FROM POSTGRESQL
# ============================================================================

def read_with_partitioning(spark, table_name, partition_column, num_partitions):
    """
    Read a table using column-based partitioning
    This is the MOST IMPORTANT optimization technique!
    """
    logger.info(f"\n📖 Reading {table_name} with {num_partitions} partitions on {partition_column}")
    
    # Get min and max values for partition column
    min_max_df = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table=f"(SELECT MIN({partition_column}) as min_val, MAX({partition_column}) as max_val FROM {table_name}) as stats",
        properties=DB_CONFIG
    )
    
    stats = min_max_df.collect()[0]
    min_val = stats["min_val"]
    max_val = stats["max_val"]
    total_range = max_val - min_val
    
    logger.info(f"   Range: {min_val} to {max_val} (total: {total_range})")
    
    # Read with partitioning
    start_time = time.time()
    
    df = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table=table_name,
        column=partition_column,
        lowerBound=min_val,
        upperBound=max_val + 1,  # +1 to include max value
        numPartitions=num_partitions,
        properties=DB_CONFIG
    )
    
    elapsed = time.time() - start_time
    
    # Get partition distribution
    partition_counts = df.withColumn("partition_id", spark_partition_id()) \
                         .groupBy("partition_id") \
                         .count() \
                         .orderBy("partition_id") \
                         .collect()
    
    logger.info(f"   ✅ Read {df.count():,} rows in {elapsed:.2f} seconds")
    logger.info(f"   📊 Actual partitions: {df.rdd.getNumPartitions()}")
    
    # Show partition distribution
    logger.info("   📈 Partition distribution:")
    for row in partition_counts[:5]:  # Show first 5
        logger.info(f"      Partition {row['partition_id']}: {row['count']:,} rows")
    if len(partition_counts) > 5:
        logger.info(f"      ... and {len(partition_counts)-5} more partitions")
    
    return df, elapsed

def read_without_partitioning(spark, table_name):
    """Read without partitioning (baseline for comparison)"""
    logger.info(f"\n📖 Reading {table_name} WITHOUT partitioning (baseline)")
    
    start_time = time.time()
    
    df = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table=table_name,
        properties=DB_CONFIG
    )
    
    elapsed = time.time() - start_time
    
    logger.info(f"   ✅ Read {df.count():,} rows in {elapsed:.2f} seconds")
    logger.info(f"   📊 Partitions: {df.rdd.getNumPartitions()}")
    
    return df, elapsed

# ============================================================================
# PART 2: COMPARE PARTITIONING STRATEGIES
# ============================================================================

def compare_partitioning_strategies(spark):
    """Compare different partitioning strategies"""
    logger.info("\n" + "=" * 70)
    logger.info("COMPARING PARTITIONING STRATEGIES")
    logger.info("=" * 70)
    
    # Use order_items table (has good numeric columns for partitioning)
    table = "order_items"
    
    # Test 1: No partitioning
    df1, time1 = read_without_partitioning(spark, table)
    
    # Test 2: With partitioning (4 partitions)
    df2, time2 = read_with_partitioning(spark, table, "order_item_id", 4)
    
    # Test 3: With partitioning (8 partitions)
    df3, time3 = read_with_partitioning(spark, table, "order_item_id", 8)
    
    # Test 4: With partitioning (16 partitions)
    df4, time4 = read_with_partitioning(spark, table, "order_item_id", 16)
    
    # Summary
    logger.info("\n" + "=" * 70)
    logger.info("PARTITIONING PERFORMANCE SUMMARY")
    logger.info("=" * 70)
    logger.info(f"{'Strategy':<20} {'Time (s)':<12} {'Speedup':<10} {'Partitions':<10}")
    logger.info("-" * 60)
    logger.info(f"{'No partitioning':<20} {time1:<12.2f} {'1.0x':<10} {df1.rdd.getNumPartitions():<10}")
    logger.info(f"{'4 partitions':<20} {time2:<12.2f} {(time1/time2):<10.1f}x {df2.rdd.getNumPartitions():<10}")
    logger.info(f"{'8 partitions':<20} {time3:<12.2f} {(time1/time3):<10.1f}x {df3.rdd.getNumPartitions():<10}")
    logger.info(f"{'16 partitions':<20} {time4:<12.2f} {(time1/time4):<10.1f}x {df4.rdd.getNumPartitions():<10}")
    
    return {
        "no_partition": (df1, time1),
        "part_4": (df2, time2),
        "part_8": (df3, time3),
        "part_16": (df4, time4)
    }


# ============================================================================
# PART 3: OPTIMIZED WRITING TO POSTGRESQL
# ============================================================================

def write_with_batching(df, table_name, batch_size):
    """Write to PostgreSQL with custom batch size"""
    logger.info(f"\n📝 Writing {table_name} with batch size {batch_size}")
    
    props_with_batch = DB_CONFIG.copy()
    props_with_batch["batchsize"] = str(batch_size)
    
    start_time = time.time()
    
    df.write.mode("overwrite") \
        .format("jdbc") \
        .option("url", DB_CONFIG["url"]) \
        .option("dbtable", f"test_{table_name}_batch{batch_size}") \
        .option("user", DB_CONFIG["user"]) \
        .option("password", DB_CONFIG["password"]) \
        .option("driver", DB_CONFIG["driver"]) \
        .option("batchsize", str(batch_size)) \
        .save()
    
    elapsed = time.time() - start_time
    logger.info(f"   ✅ Wrote {df.count():,} rows in {elapsed:.2f} seconds")
    
    return elapsed

def compare_batch_sizes(spark, test_df):
    """Compare different batch sizes for writing"""
    logger.info("\n" + "=" * 70)
    logger.info("COMPARING BATCH SIZES FOR WRITING")
    logger.info("=" * 70)
    
    batch_sizes = [1000, 5000, 10000, 50000]
    results = {}
    
    for batch_size in batch_sizes:
        elapsed = write_with_batching(test_df, "batch_test", batch_size)
        results[batch_size] = elapsed
    
    # Summary
    logger.info("\n" + "=" * 70)
    logger.info("BATCH SIZE PERFORMANCE SUMMARY")
    logger.info("=" * 70)
    logger.info(f"{'Batch Size':<15} {'Time (s)':<12} {'Rows/sec':<15}")
    logger.info("-" * 50)
    
    base_time = results[1000]  # Use smallest batch as baseline
    row_count = test_df.count()
    
    for batch_size, elapsed in results.items():
        rows_per_sec = row_count / elapsed
        logger.info(f"{batch_size:<15} {elapsed:<12.2f} {rows_per_sec:<15,.0f}")

# ============================================================================
# PART 4: PREDICATE PUSHDOWN (Filter in PostgreSQL)
# ============================================================================

def compare_filter_pushdown(spark):
    """Compare filtering in Spark vs PostgreSQL"""
    logger.info("\n" + "=" * 70)
    logger.info("COMPARING FILTER PUSHDOWN")
    logger.info("=" * 70)
    
    # Method 1: Read all, filter in Spark
    logger.info("\n🔍 Method 1: Read ALL data, filter in Spark")
    
    start = time.time()
    df_all = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table="order_items",
        properties=DB_CONFIG
    )
    filtered_spark = df_all.filter(col("quantity") > 3)
    count_spark = filtered_spark.count()
    time_spark = time.time() - start
    logger.info(f"   ✅ Found {count_spark} rows in {time_spark:.2f} seconds")
    
    # Method 2: Push filter to PostgreSQL (FASTER!)
    logger.info("\n🔍 Method 2: Push filter to PostgreSQL")
    
    start = time.time()
    df_pushdown = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table="(SELECT * FROM order_items WHERE quantity > 3) as filtered",
        properties=DB_CONFIG
    )
    count_pushdown = df_pushdown.count()
    time_pushdown = time.time() - start
    logger.info(f"   ✅ Found {count_pushdown} rows in {time_pushdown:.2f} seconds")
    
    # Summary
    logger.info("\n" + "=" * 70)
    logger.info("FILTER PUSHDOWN SUMMARY")
    logger.info("=" * 70)
    logger.info(f"{'Method':<25} {'Time (s)':<12} {'Speedup':<10}")
    logger.info("-" * 50)
    logger.info(f"{'Filter in Spark':<25} {time_spark:<12.2f} {'1.0x':<10}")
    logger.info(f"{'Pushdown to PostgreSQL':<25} {time_pushdown:<12.2f} {(time_spark/time_pushdown):<10.1f}x")

# ============================================================================
# PART 5: CREATE TEST DATA FOR OPTIMIZATION
# ============================================================================

def create_test_data(spark):
    """Create a larger test dataset for meaningful benchmarks"""
    logger.info("\n" + "=" * 70)
    logger.info("CREATING TEST DATASET FOR BENCHMARKS")
    logger.info("=" * 70)
    
    # Read existing data
    order_items = spark.read.jdbc(
        url=DB_CONFIG["url"],
        table="order_items",
        properties=DB_CONFIG
    )
    
    # Multiply the data to create larger dataset (5x)
    test_data = order_items
    for i in range(1, 5):
        multiplied = order_items.withColumn("order_item_id", col("order_item_id") + (i * 10000))
        test_data = test_data.union(multiplied)
    
    test_data = test_data.withColumn("order_item_id", monotonically_increasing_id())
    
    logger.info(f"   ✅ Created test dataset with {test_data.count():,} rows")
    return test_data



In [3]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================
def main():
    """Main execution function"""
    logger.info("=" * 70)
    logger.info("SPARK-POSTGRESQL OPTIMIZATION TUTORIAL")
    logger.info("=" * 70)
    
    spark = init_spark()
    
    try:
        # Part 1: Compare partitioning strategies
        partitioning_results = compare_partitioning_strategies(spark)
        
        # Part 2: Create larger test data
        test_data = create_test_data(spark)
        
        # Part 3: Compare batch sizes
        compare_batch_sizes(spark, test_data.limit(10000))  # Use 10k rows for write tests
        
        # Part 4: Compare filter pushdown
        compare_filter_pushdown(spark)
        
        # Final optimization recommendations
        logger.info("\n" + "=" * 70)
        logger.info("OPTIMIZATION BEST PRACTICES")
        logger.info("=" * 70)
        logger.info("""
📌 READING OPTIMIZATION:
   • Use column-based partitioning for large tables
   • Choose partition count = number of cores * 2-3
   • Partition column should have even distribution
   • Use predicate pushdown (filter in SQL, not Spark)

📌 WRITING OPTIMIZATION:
   • Batch size: 5000-10000 is usually optimal
   • Use 'overwrite' with truncate for full reloads
   • Consider 'append' for incremental loads
   • Disable auto-commit for better performance

📌 GENERAL TIPS:
   • Monitor Spark UI at http://localhost:4040
   • Use EXPLAIN to see execution plans
   • Analyze table statistics in PostgreSQL
   • Consider connection pooling for production
        """)
        
    finally:
        spark.stop()
        logger.info("\n✅ Spark session stopped")

if __name__ == "__main__":
    main()

2026-03-14 08:34:20,797 - INFO - ======================================================================
2026-03-14 08:34:20,799 - INFO - SPARK-POSTGRESQL OPTIMIZATION TUTORIAL
2026-03-14 08:34:20,800 - INFO - ======================================================================
26/03/14 08:34:25 WARN Utils: Your hostname, odinsbeard-VirtualBox resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/03/14 08:34:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/14 08:34:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2026-03-14 08:34:29,803 - INFO - ✅ Spark session created (version: 3.5.0)
2026-03-14 08:34:29,804 - INFO - 
2026-03-14 08:34:29,805 - INFO - COMPARING PARTITIONING STRATEGIES
2026-03-14 08:34:29,806 - IN